## Symbolic Detection of Simile Structures

To complement neural inference, a rule-based symbolic detector is implemented
based on principles from Arabic rhetorical theory (علم البيان).

The symbolic module evaluates:

* Explicit simile particles (e.g., كأن، كما)
* Comparison verbs and nominal similarity markers
* Prefix-based simile morphology
* Contextual syntactic structure patterns
* Logical comparison contexts that reduce rhetorical likelihood

The detector produces:

* A confidence score
* A traceable linguistic explanation

In [2]:
from dataclasses import dataclass, field


In [ ]:

SIMILE_PARTICLES_STRONG = {"كأن","كأنما"}
SIMILE_PARTICLES_WEAK = {"كما"}

SIMILE_VERBS = {"يشبه","شبه","يماثل","يضارع"}
SIMILE_NOUNS = {"مثل","شبيه","نظير"}

FALSE_CONTEXT = {"كما أن","كما كان","كما ينبغي"}

def tokenize(text):
    return text.split()

def is_probable_noun(word):
    return word.startswith("ال") or word.endswith("ة")

def has_prefix(word):
    return word.startswith("ك") and len(word)>2


In [11]:
@dataclass
class Evidence:
    confidence: float = 0.0
    trace: list = field(default_factory=list)

In [12]:
def symbolic_detector(sentence):

    words = tokenize(sentence)
    ev = Evidence()

    for i,w in enumerate(words):

        if w in SIMILE_PARTICLES_STRONG:
            ev.confidence += 0.7
            ev.trace.append(f"Strong particle '{w}'")

        if w in SIMILE_PARTICLES_WEAK:
            ev.confidence += 0.3
            ev.trace.append(f"Weak particle '{w}'")

        if w in SIMILE_VERBS:
            ev.confidence += 0.5
            ev.trace.append(f"Comparison verb '{w}'")

        if w in SIMILE_NOUNS:
            ev.confidence += 0.25
            ev.trace.append(f"Nominal marker '{w}'")

        if has_prefix(w):
            ev.confidence += 0.2
            ev.trace.append(f"Prefix simile form '{w}'")

        if w in SIMILE_PARTICLES_STRONG.union(SIMILE_PARTICLES_WEAK):
            if i>0 and i<len(words)-1:
                left = words[i-1]
                right = words[i+1]
                if is_probable_noun(left) and is_probable_noun(right):
                    ev.confidence += 0.4
                    ev.trace.append(f"Full structure {left} {w} {right}")

    for bad in FALSE_CONTEXT:
        if bad in sentence:
            ev.confidence -= 0.4
            ev.trace.append("Logical comparison context detected")

    ev.confidence = max(ev.confidence,0)
    return ev



In [ ]:
def neuro_symbolic_classifier(sentence, bert_prob):

    ev = symbolic_detector(sentence)
   # Neuro-Symbolic Fusion
    score = 0.6*bert_prob + 0.4*ev.confidence

    explanation = [
        f"BERT probability = {bert_prob:.3f}",
        f"Symbolic confidence = {ev.confidence:.3f}",
        f"Hybrid score = {score:.3f}"
    ]

    explanation.extend(ev.trace)

    label = 1 if score>=0.5 else 0
    return label, explanation

In [ ]:
bert_probs = model.predict(
    {"input_ids": X_test_ids, "attention_mask": X_test_mask}
).flatten()

final_preds = []
all_exp = []

for s,p in zip(X_test, bert_probs):
    lab, exp = neuro_symbolic_classifier(s,p)
    final_preds.append(lab)
    all_exp.append((s,p,exp))

cm = confusion_matrix(y_test, final_preds)
print(cm)
print(classification_report(y_test, final_preds))


# print sample explanations
for i in range(5):
    print("\n================")
    print("Sentence:", all_exp[i][0])
    print("BERT:", all_exp[i][1])
    for e in all_exp[i][2]:
        print("-", e)